# Marketing Science Empirical Analysis — Step-by-Step Story Progression

Evaluation of `jaredpalmer/kev-0.8b`, `kev-4b`, and `kev-9b` across the paper's four-step narrative, using one validation benchmark and two held-out out-of-domain test benchmarks:

### Dataset Roles
- **Training studies**:
  - `Wu 2025` (flight conjoint): either **8 published OLS slopes** (`Row 2`, `Row 4`) or **`18,591` respondent microdata choices** (`Row 2b`, `Row 5`).
  - `Berke` (consumer choice microdata): respondent choices (`Row 3`, `Row 4`, `Row 5`).
- **Validation & checkpoint selection (`best.pt`)**:
  - `Kriner 2026` (flight conjoint, 12 terms): selects `best.pt` via lowest `kriner2026/beta_mse` and translates slopes into USD Willingness-To-Pay error (`wtp_mae_$`).
- **Held-out out-of-domain test sets (`test_best/`, never used in training or checkpoint selection)**:
  - `Grummon 2025` (food warning labels, 5 terms): `grummon_mse`, `grummon_kl`.
  - `COVID` (`Kreps et al. 2020` vaccine conjoint, 11 terms; `1,971` respondents, `9,855` tasks): `covid_mse`, `covid_kl`.

### Narrative Roadmap (Cell by Cell)
- **Step 0 — Run & `test_best` Inventory**: Track completed HPC runs (`runs-rorqual/`) and local `test_best/` evaluations (`Grummon` + `COVID`).
- **Step 1 — Untrained Off-the-Shelf LLMs Are Poor Choice Simulators (`Row 1`)**: Establish zero-shot econometric bias, weak price sensitivity, and poor calibration across all three benchmarks.
- **Step 2 — Fine-Tuning on Microdata Works & Defines the Oracles (`Row 2b`, `Row 3`, `Row 5`)**: Show that respondent-level CE training generalizes out-of-domain and defines both the **study-matched single-study oracle (`Row 2b`: `Wu` CE)** and the **global full-information oracle (`Row 5`: `Berke + Wu` CE)**.
- **Step 3 — Learning from Published $\beta^*$ Tables Without Microdata (`Row 2`: Population $\beta$-Loss + Forward KL Anchor)**: Show that unanchored $\beta$-loss (`No-KL`) collapses out-of-domain on `Grummon` and `COVID`, whereas forward KL anchoring (`KL=0.005`) prevents collapse and recovers microdata-like preferences from just **8 published numbers** (`Step 3a` table + `Step 3b` trajectories & $\alpha$ calibration).
- **Step 4 — Hybrid Training: Mixing Accessible Microdata with Published $\beta^*$ Tables (`Row 3` $\to$ `Row 4` vs `Row 5` Oracle)**: Show that augmenting accessible microdata (`Berke`) with `Wu`'s published table closes the gap to the full `Berke + Wu` microdata oracle (`Row 5`).
- **Step 5 — Economic Translation & Unified Master Progression**: Attribute-level USD Flight WTP (`Kriner`), vaccine preference slopes (`COVID`), and the end-to-end `Row 1` $\to$ `Row 5` summary.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"
PAPER_RUNS_ROOT = PAPER_DIR / "runs-rorqual"
LOCAL_PAPER_RUNS_ROOT = PAPER_DIR / "runs"
LEGACY_RUNS_ROOT = BETA_TRAINING_DIR / "runs"
TEST_BEST_SUBDIR = "test_best"

os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from beta_ce_population.trainer import load_ce_population_run
from beta_population.trainer import load_population_run
from beta_train import runs as run_store
from beta_train.trainer import load_run
from paper_beta import extract_step0_from_run_dir, load_paper_run

MODEL_SLUGS = ("kev-0.8b", "kev-4b", "kev-9b")
VALIDATORS = ("kriner2026", "grummon2025")
KRINER_PRICE_LEVELS = {"Price: $275": 25.0, "Price: $300": 50.0, "Price: $325": 75.0}

REGIME_LABELS = {
    "row1_base_untrained": "Row 1: Untrained Base",
    "row2_wu_table_anchored": "Row 2: Wu Table (KL=0.005)",
    "row2_ablation_wu_table_no_kl": "Row 2: Wu Table (No-KL)",
    "row2b_wu_ce_oracle": "Row 2b: Wu Microdata CE (Study Oracle)",
    "row3_berke_ce_only": "Row 3: Berke Microdata CE",
    "row4_hybrid_berke_ce_wu_table_anchored": "Row 4: Berke CE + Wu Table (KL=0.005)",
    "row4_hybrid_berke_ce_wu_table_no_kl": "Row 4: Berke CE + Wu Table (No-KL)",
    "row5_full_berke_wu_ce_oracle": "Row 5: Berke + Wu Full CE (Full Oracle)",
}
SHORT_COL_LABELS = {
    "row1_base_untrained": "Row 1: Base",
    "row2_ablation_wu_table_no_kl": "Row 2: Wu Table (No-KL)",
    "row2_wu_table_anchored": "Row 2: Wu Table (KL)",
    "row2b_wu_ce_oracle": "Row 2b: Wu CE",
    "row3_berke_ce_only": "Row 3: Berke CE",
    "row4_hybrid_berke_ce_wu_table_no_kl": "Row 4: Hybrid (No-KL)",
    "row4_hybrid_berke_ce_wu_table_anchored": "Row 4: Hybrid (KL)",
    "row5_full_berke_wu_ce_oracle": "Row 5: Full CE",
}
ALL_SLUGS = tuple(REGIME_LABELS.keys())
TRAINED_SLUGS = tuple(k for k in ALL_SLUGS if k != "row1_base_untrained")

ROUND_MAP = {
    "kriner_red_%": 1,
    "grummon_red_%": 1,
    "covid_red_%": 1,
    "price_dPr_$100": 2,
    "kriner_mse": 5,
    "kriner_gap_%": 1,
    "wtp_mae_$": 2,
    "kriner_kl": 3,
    "grummon_mse": 5,
    "grummon_gap_%": 1,
    "grummon_kl": 3,
    "covid_mse": 5,
    "covid_gap_%": 1,
    "covid_kl": 3,
}


def kriner_wtp_from_series(beta_series: pd.Series) -> pd.Series:
    """Fit linear price slope across [0, 25, 50, 75] and return implied USD WTP per attribute."""
    deltas = np.array([0.0, 25.0, 50.0, 75.0], dtype=np.float64)
    effects = np.array(
        [0.0, float(beta_series["Price: $275"]), float(beta_series["Price: $300"]), float(beta_series["Price: $325"])],
        dtype=np.float64,
    )
    cx = deltas - deltas.mean()
    cy = effects - effects.mean()
    slope_per_dollar = float((cx @ cy) / (cx @ cx))
    if slope_per_dollar >= 0:
        raise ValueError(f"Non-negative price slope {slope_per_dollar:.6f}; dollar WTP is ill-defined")
    non_price = [t for t in beta_series.index if t not in KRINER_PRICE_LEVELS]
    wtp = {term: -float(beta_series[term]) / slope_per_dollar for term in non_price}
    wtp["Price Slope (dPr / $100)"] = slope_per_dollar * 100.0
    return pd.Series(wtp)


def wtp_mae_usd(kriner_beta_df: pd.DataFrame) -> float:
    if kriner_beta_df.index.name != "term":
        raise ValueError(f"Expected beta table indexed by 'term', got {kriner_beta_df.index.name!r}")
    human_wtp = kriner_wtp_from_series(kriner_beta_df["human_pop_beta"])
    model_wtp = kriner_wtp_from_series(kriner_beta_df["model_pop_beta"])
    return float((model_wtp.iloc[:-1] - human_wtp.iloc[:-1]).abs().mean())


def load_test_best_entry(run_or_row1_dir: Path, expected_step: int) -> dict | None:
    """Load `test_best/` (`grummon2025` + `covid`) when present and verify checkpoint_step alignment."""
    test_dir = run_or_row1_dir / TEST_BEST_SUBDIR
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return None
    metrics_csv = test_dir / "metrics.csv"
    betas_csv = test_dir / "betas.csv"
    if not metrics_csv.exists() or not betas_csv.exists():
        raise FileNotFoundError(f"Incomplete test_best directory in {test_dir}")

    test_info = json.loads(test_json.read_text(encoding="utf-8"))
    actual_step = int(test_info["checkpoint_step"])
    if actual_step != expected_step:
        raise ValueError(f"Stale test_best in {test_dir}: checkpoint_step={actual_step}, expected {expected_step}")

    metrics_df = pd.read_csv(metrics_csv).set_index("dataset")
    for ds in ("grummon2025", "covid"):
        if ds not in metrics_df.index:
            raise ValueError(f"Dataset {ds!r} missing from {metrics_csv}")

    betas_df = pd.read_csv(betas_csv)
    required_cols = {"validator", "term", "human_pop_beta", "model_pop_beta"}
    if not required_cols.issubset(betas_df.columns):
        raise ValueError(f"Missing columns in {betas_csv}: expected {required_cols}, got {set(betas_df.columns)}")

    covid_betas = betas_df[betas_df["validator"] == "covid"].set_index("term")
    grummon_betas = betas_df[betas_df["validator"] == "grummon2025"].set_index("term")
    return {
        "metrics_df": metrics_df,
        "covid_betas": covid_betas,
        "grummon_betas": grummon_betas,
    }


print(f"PAPER_RUNS_ROOT : {PAPER_RUNS_ROOT}")

---
## Step 0 — Run & `test_best` Completion Inventory

Single status matrix across `kev-0.8b`, `kev-4b`, and `kev-9b` showing each regime's training state (`finished`, `interrupted`, `starting`, or `—`) alongside whether `04-test-best-*.ipynb` has evaluated `best.pt` on `Grummon 2025` + `COVID` (`[test_best ✓]` vs `[test_best pending]`).

In [ ]:
def collect_model_regimes(
    model_slug: str,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    model_dir = PAPER_RUNS_ROOT / model_slug
    existing_dirs = [model_dir / s for s in TRAINED_SLUGS if (model_dir / s / "run.json").exists()]
    if len(existing_dirs) == 0:
        raise FileNotFoundError(f"No canonical regime runs found in {model_dir}")

    step0 = extract_step0_from_run_dir(existing_dirs[0], VALIDATORS)
    step0_kriner_betas = step0["betas_by_validator"]["kriner2026"]
    step0_wtp = kriner_wtp_from_series(step0_kriner_betas["model_pop_beta"])
    kriner_betas_by_slug: dict[str, pd.DataFrame] = {"row1_base_untrained": step0_kriner_betas}
    covid_betas_by_slug: dict[str, pd.DataFrame] = {}

    row1_tb = load_test_best_entry(model_dir / "row1_base_untrained", expected_step=0)
    if row1_tb is not None:
        g0_tb = float(row1_tb["metrics_df"].loc["grummon2025", "beta_mse"])
        g0_val = step0["metrics"]["grummon2025/beta_mse"]
        if abs(g0_tb - g0_val) > 2e-4:
            raise ValueError(f"{model_slug} Row 1 Grummon mismatch: val={g0_val:.6f} vs test_best={g0_tb:.6f}")
        row1_grummon_mse = g0_tb
        row1_grummon_kl = float(row1_tb["metrics_df"].loc["grummon2025", "kl_loss"])
        row1_covid_mse = float(row1_tb["metrics_df"].loc["covid", "beta_mse"])
        row1_covid_kl = float(row1_tb["metrics_df"].loc["covid", "kl_loss"])
        covid_betas_by_slug["row1_base_untrained"] = row1_tb["covid_betas"]
        row1_progress = "step 0 [test_best ✓]"
    else:
        row1_grummon_mse = step0["metrics"]["grummon2025/beta_mse"]
        row1_grummon_kl = step0["metrics"]["grummon2025/kl_loss"]
        row1_covid_mse = np.nan
        row1_covid_kl = np.nan
        row1_progress = "step 0 [test_best pending]"

    rows = [
        {
            "slug": "row1_base_untrained",
            "regime": REGIME_LABELS["row1_base_untrained"],
            "status": "step0",
            "progress": row1_progress,
            "best_step": 0,
            "price_dPr_$100": float(step0_wtp["Price Slope (dPr / $100)"]),
            "kriner_mse": step0["metrics"]["kriner2026/beta_mse"],
            "wtp_mae_$": wtp_mae_usd(step0_kriner_betas),
            "kriner_kl": step0["metrics"]["kriner2026/kl_loss"],
            "grummon_mse": row1_grummon_mse,
            "grummon_kl": row1_grummon_kl,
            "covid_mse": row1_covid_mse,
            "covid_kl": row1_covid_kl,
        }
    ]

    for slug in TRAINED_SLUGS:
        run_dir = model_dir / slug
        if not (run_dir / "run.json").exists():
            rows.append(
                {
                    "slug": slug,
                    "regime": REGIME_LABELS[slug],
                    "status": "not_run_yet",
                    "progress": "—",
                    "best_step": np.nan,
                    "price_dPr_$100": np.nan,
                    "kriner_mse": np.nan,
                    "wtp_mae_$": np.nan,
                    "kriner_kl": np.nan,
                    "grummon_mse": np.nan,
                    "grummon_kl": np.nan,
                    "covid_mse": np.nan,
                    "covid_kl": np.nan,
                }
            )
            continue

        run_info = run_store.read_run_json(run_dir)
        raw_status = run_info["status"]
        result = load_paper_run(run_dir)
        steps_done = len(result.step_df)
        best_step = int(result.best["step"])

        if steps_done == 0:
            rows.append(
                {
                    "slug": slug,
                    "regime": REGIME_LABELS[slug],
                    "status": f"{raw_status} (0 steps)",
                    "progress": f"{raw_status} (0 steps)",
                    "best_step": np.nan,
                    "price_dPr_$100": np.nan,
                    "kriner_mse": np.nan,
                    "wtp_mae_$": np.nan,
                    "kriner_kl": np.nan,
                    "grummon_mse": np.nan,
                    "grummon_kl": np.nan,
                    "covid_mse": np.nan,
                    "covid_kl": np.nan,
                }
            )
            continue

        k_metrics, k_betas = result.best_validation["kriner2026"]
        k_wtp = kriner_wtp_from_series(k_betas["model_pop_beta"])
        kriner_betas_by_slug[slug] = k_betas

        tb_entry = load_test_best_entry(run_dir, expected_step=best_step)
        if tb_entry is not None:
            g_mse = float(tb_entry["metrics_df"].loc["grummon2025", "beta_mse"])
            g_kl = float(tb_entry["metrics_df"].loc["grummon2025", "kl_loss"])
            if "grummon2025" in result.best_validation:
                g_val = float(result.best_validation["grummon2025"][0]["beta_mse"])
                if abs(g_mse - g_val) > 2e-4:
                    raise ValueError(f"{model_slug}/{slug} Grummon mismatch: val={g_val:.6f} vs test_best={g_mse:.6f}")
            c_mse = float(tb_entry["metrics_df"].loc["covid", "beta_mse"])
            c_kl = float(tb_entry["metrics_df"].loc["covid", "kl_loss"])
            covid_betas_by_slug[slug] = tb_entry["covid_betas"]
            status_label = raw_status if raw_status == "finished" else f"{raw_status} ({steps_done} steps)"
            tb_tag = "[test_best ✓]"
        else:
            c_mse = np.nan
            c_kl = np.nan
            tb_tag = "[test_best pending]"
            if "grummon2025" in result.best_validation:
                g_mse = float(result.best_validation["grummon2025"][0]["beta_mse"])
                g_kl = float(result.best_validation["grummon2025"][0]["kl_loss"])
                status_label = raw_status if raw_status == "finished" else f"{raw_status} ({steps_done} steps)"
            else:
                g_valid = result.val_history_df["grummon2025/beta_mse"].dropna()
                g_step = int(g_valid.index[np.abs(g_valid.index.to_numpy() - best_step).argmin()])
                g_mse = float(g_valid.loc[g_step])
                g_kl = float(result.val_history_df.loc[g_step, "grummon2025/kl_loss"])
                status_label = f"{raw_status} ({steps_done} steps, g@{g_step})"

        progress_str = f"{raw_status} (@{best_step}/{steps_done}) {tb_tag}"
        rows.append(
            {
                "slug": slug,
                "regime": REGIME_LABELS[slug],
                "status": status_label,
                "progress": progress_str,
                "best_step": best_step,
                "price_dPr_$100": float(k_wtp["Price Slope (dPr / $100)"]),
                "kriner_mse": float(k_metrics["beta_mse"]),
                "wtp_mae_$": wtp_mae_usd(k_betas),
                "kriner_kl": float(k_metrics["kl_loss"]),
                "grummon_mse": g_mse,
                "grummon_kl": g_kl,
                "covid_mse": c_mse,
                "covid_kl": c_kl,
            }
        )

    return pd.DataFrame(rows).set_index("slug"), kriner_betas_by_slug, covid_betas_by_slug


regimes_by_model: dict[str, pd.DataFrame] = {}
kriner_betas_by_model: dict[str, dict[str, pd.DataFrame]] = {}
covid_betas_by_model: dict[str, dict[str, pd.DataFrame]] = {}
for model_slug in MODEL_SLUGS:
    regimes_by_model[model_slug], kriner_betas_by_model[model_slug], covid_betas_by_model[model_slug] = (
        collect_model_regimes(model_slug)
    )

status_grid = pd.DataFrame(
    {m: regimes_by_model[m].loc[list(ALL_SLUGS), "progress"].values for m in MODEL_SLUGS},
    index=[REGIME_LABELS[s] for s in ALL_SLUGS],
)
status_grid.index.name = "regime"
display(status_grid)

---
## Step 1 of the Story — Untrained Off-the-Shelf LLMs Are Poor Choice Simulators (`Row 1`)

**Claim**: Off-the-shelf instruction-tuned LLMs (`kev-0.8b`, `kev-4b`, `kev-9b`) cannot serve directly as quantitative consumer choice simulators:
1. **Severe price under-reaction**: On `Kriner 2026` flights, human choice share drops by **`-0.41` per `$100`** (`dPr / $100`), whereas untrained LLMs react at only **`-0.03` (`0.8B`)**, **`-0.21` (`4B`)**, and **`-0.17` (`9B`)**—inflating implied USD Willingness-To-Pay (`wtp_mae_$`) to **`$102.58` (`0.8B`)**, **`$28.33` (`4B`)**, and **`$20.71` (`9B`)**.
2. **Cross-domain slope bias & miscalibration**: Population slope MSE and choice-share KL remain elevated on both held-out test domains (`Grummon 2025` food warning labels and `COVID / Kreps 2020` vaccine choices).

In [ ]:
human_kriner_wtp = kriner_wtp_from_series(kriner_betas_by_model["kev-0.8b"]["row1_base_untrained"]["human_pop_beta"])
human_price_slope_100 = float(human_kriner_wtp["Price Slope (dPr / $100)"])

step1_rows = []
for model_slug in MODEL_SLUGS:
    r1 = regimes_by_model[model_slug].loc["row1_base_untrained"]
    step1_rows.append(
        {
            "model": model_slug,
            "human_dPr_$100": human_price_slope_100,
            "price_dPr_$100": r1["price_dPr_$100"],
            "kriner_mse": r1["kriner_mse"],
            "wtp_mae_$": r1["wtp_mae_$"],
            "kriner_kl": r1["kriner_kl"],
            "grummon_mse": r1["grummon_mse"],
            "grummon_kl": r1["grummon_kl"],
            "covid_mse": r1["covid_mse"],
            "covid_kl": r1["covid_kl"],
        }
    )

step1_df = pd.DataFrame(step1_rows).set_index("model").round({**ROUND_MAP, "human_dPr_$100": 2})
print("=== Step 1: Untrained Base LLM (`Row 1`, Zero-Shot) Across Validation & Held-Out Test Sets ===")
display(step1_df)

---
## Step 2 of the Story — Training on Choice Microdata Works Out-of-Domain & Defines the Oracles (`Row 2b`, `Row 3`, `Row 5`)

**Claim (Empirical Benchmark / Contribution 1)**: Fine-tuning on respondent-level choice microdata via cross-entropy (`CE`) fixes price sensitivity on `Kriner` **and transfers zero-shot** to completely unseen product and policy domains (`Grummon` food warning labels and `COVID` vaccines—cutting `COVID` slope MSE by `69%–81%` across `0.8B` and `4B`).

This establishes **two microdata oracle ceilings** for the rest of the paper:
- **`Row 2b: Wu Microdata CE` (Single-Study Oracle)**: Trained on `Wu`'s `18,591` respondent choices—the exact information-complete counterpart to `Row 2` (`Wu`'s 8 published OLS slopes) in **Step 3**.
- **`Row 5: Berke + Wu Full CE` (Global Full-Information Oracle)**: Trained on pooled `Berke + Wu` microdata—the global ceiling across the entire study and the exact counterpart to hybrid `Row 4` in **Step 4**.

**The Data-Access Bottleneck**: Respondent choice microdata (`y`) is proprietary, restricted, or unpublished for the vast majority of marketing and economics papers—whereas **aggregate regression tables ($\beta^*$) are published everywhere**.

In [ ]:
def build_step_table(
    models: tuple[str, ...],
    slugs: tuple[str, ...],
    oracle_slug: str | None,
) -> pd.DataFrame:
    """Build comparison table. If oracle_slug is None, compute % MSE reduction vs Row 1; else % of Row 1 -> Oracle gap closed."""
    frames = []
    pct_suffix = "red_%" if oracle_slug is None else "gap_%"
    for model_slug in models:
        sub = regimes_by_model[model_slug].loc[list(slugs)].copy()
        sub["best_step"] = sub["best_step"].round(0).astype("Int64")
        for prefix in ("kriner", "grummon", "covid"):
            col = f"{prefix}_mse"
            pct_col = f"{prefix}_{pct_suffix}"
            base_val = sub.loc["row1_base_untrained", col]
            if oracle_slug is None:
                if pd.isna(base_val):
                    sub[pct_col] = np.nan
                else:
                    if float(base_val) == 0.0:
                        raise ValueError(f"Zero base {col} on {model_slug}")
                    sub[pct_col] = 100.0 * (float(base_val) - sub[col]) / float(base_val)
            else:
                oracle_val = sub.loc[oracle_slug, col]
                if pd.isna(base_val) or pd.isna(oracle_val):
                    sub[pct_col] = np.nan
                else:
                    denom = float(base_val) - float(oracle_val)
                    if denom == 0.0:
                        raise ValueError(f"Zero {col} gap between base and {oracle_slug} on {model_slug}")
                    sub[pct_col] = 100.0 * (float(base_val) - sub[col]) / denom

        sub["model"] = model_slug
        ordered_cols = [
            "model",
            "regime",
            "status",
            "best_step",
            "kriner_mse",
            f"kriner_{pct_suffix}",
            "wtp_mae_$",
            "kriner_kl",
            "grummon_mse",
            f"grummon_{pct_suffix}",
            "grummon_kl",
            "covid_mse",
            f"covid_{pct_suffix}",
            "covid_kl",
        ]
        frames.append(sub[ordered_cols].set_index(["model", "regime"]))
    return pd.concat(frames).round(ROUND_MAP)


STEP2_SLUGS = (
    "row1_base_untrained",
    "row2b_wu_ce_oracle",
    "row3_berke_ce_only",
    "row5_full_berke_wu_ce_oracle",
)
print("=== Step 2: Microdata SFT vs Untrained Base (*_red_% = % MSE Reduction Relative to Row 1) ===")
step2_df = build_step_table(("kev-0.8b", "kev-4b"), STEP2_SLUGS, oracle_slug=None)
display(step2_df)

---
## Step 3a of the Story — Learning from Published $\beta^*$ Tables Without Microdata: Population $\beta$-Loss + Forward KL Anchor (`Row 2`)

**Claim**: Can we align a choice simulator using **only 8 published OLS coefficients** from `Wu 2025` (`Row 2`), with zero respondent choices ($y$)?
1. **Unanchored $\beta$-loss (`Row 2: No-KL`, $\alpha = 0$) is underdetermined and collapses out-of-domain**: Matching 8 slopes with millions of LoRA parameters sharpens logits arbitrarily. On `kev-0.8b`, `No-KL` fits `Kriner` flights (`0.00668`), **but degrades held-out `Grummon` `5×` worse than untrained base (`0.00546 -> 0.02690`) and held-out `COVID` `+70%` worse (`0.01207 -> 0.02046`)**; on `kev-9b`, `Grummon` also degrades worse than base (`0.02272 -> 0.02833`).
2. **Frozen-reference forward KL anchoring ($\alpha\,\mathrm{KL}(p_0 \,\|\, p_\theta)$, `Row 2: KL=0.005`) solves the ill-posedness**:
   - On `kev-0.8b` & `kev-9b`, `KL=0.005` achieves the same strong `Kriner` flight recovery (`0.00687` and `0.00154`, reducing WTP MAE from `$102.58 -> $18.47` and `$20.71 -> $9.05`) while eliminating the out-of-domain blow-up on `Grummon` and `COVID`.
   - On `kev-4b`, **8 published flight slopes (`Row 2: KL=0.005`) improve every single benchmark simultaneously** over both Untrained Base and `No-KL`—closing **`90.3%`** of the `Row 2b` microdata oracle gap on `Kriner` (`0.00410 -> 0.00195`), **`84.2%`** on `Grummon` (`0.02110 -> 0.00633`), and **`43.1%`** on `COVID` (`0.00908 -> 0.00591`, with `COVID` KL dropping `1.525 -> 1.202`)!

In [ ]:
STEP3_SLUGS = (
    "row1_base_untrained",
    "row2_ablation_wu_table_no_kl",
    "row2_wu_table_anchored",
    "row2b_wu_ce_oracle",
    "row5_full_berke_wu_ce_oracle",
)
print("=== Step 3a: Single-Study Published Table (`Wu` 8 Slopes) vs Study-Matched Oracle (`Row 2b` = 100% Gap) ===")
step3_df = build_step_table(MODEL_SLUGS, STEP3_SLUGS, "row2b_wu_ce_oracle")
display(step3_df)

---
## Step 3b of the Story — Mechanism: KL-Anchor Training Dynamics & $\alpha$ Calibration

- **Figure 1a (Trajectories across `0.8B`, `4B`, `9B`)**: Tracks validation (`Kriner` MSE) and held-out (`Grummon` MSE & Human KL) over optimization steps for `Row 2 (KL=0.005)` vs `Row 2 (No-KL)`. Without the anchor, held-out `Grummon` MSE and KL diverge steadily as logits sharpen.
- **Pre-Flight $\alpha$ Sweeps (`kl_anchor_800m`, `kl_anchor_9b`)**: Demonstrates why $\alpha = 0.005$ sits at the interior optimum—large enough to bound drift from $p_0$ (`kl_p0_tail`), small enough not to pin the model to `Row 1`'s weak price slope (`$\alpha \ge 0.02$`).

In [ ]:
PANEL_SPECS = [
    ("kriner2026/beta_mse", "Kriner Beta MSE (Selection)"),
    ("grummon2025/beta_mse", "Grummon Beta MSE (Held-Out Test)"),
    ("grummon2025/kl_loss", "Grummon Human KL (Calibration)"),
]


def plot_regime_grid(models: tuple[str, ...], slugs: tuple[str, ...], suptitle: str) -> None:
    fig, axes = plt.subplots(len(models), 3, figsize=(15.0, 3.3 * len(models)), squeeze=False)
    for r_idx, model_slug in enumerate(models):
        base_row = regimes_by_model[model_slug].loc["row1_base_untrained"]
        base_lines = {
            "kriner2026/beta_mse": float(base_row["kriner_mse"]),
            "grummon2025/beta_mse": float(base_row["grummon_mse"]),
            "grummon2025/kl_loss": float(base_row["grummon_kl"]),
        }
        for slug in slugs:
            run_dir = PAPER_RUNS_ROOT / model_slug / slug
            if not (run_dir / "run.json").exists():
                continue
            result = load_paper_run(run_dir)
            if len(result.step_df) == 0:
                continue
            label = SHORT_COL_LABELS[slug]
            for c_idx, (col, col_title) in enumerate(PANEL_SPECS):
                ax = axes[r_idx, c_idx]
                series = result.val_history_df[col].dropna()
                ax.plot(series.index, series.values, marker="o", markersize=2.8, linewidth=1.6, label=label)
                if r_idx == 0:
                    ax.set_title(col_title, fontsize=10.5, fontweight="bold")
                if c_idx == 0:
                    ax.set_ylabel(model_slug, fontsize=10.5, fontweight="bold")
                if r_idx == len(models) - 1:
                    ax.set_xlabel("Step")
        for c_idx, (col, _) in enumerate(PANEL_SPECS):
            ax = axes[r_idx, c_idx]
            ax.axhline(base_lines[col], color="gray", linestyle=":", linewidth=1.1, label="Row 1: Base")
            ax.grid(alpha=0.25)
            if c_idx == 2:
                ax.legend(fontsize=8, loc="best")
    fig.suptitle(suptitle, fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()


plot_regime_grid(
    MODEL_SLUGS,
    ("row2_wu_table_anchored", "row2_ablation_wu_table_no_kl"),
    "Figure 1a — Step 3 Trajectories (Wu Table Only): KL Anchor (alpha=0.005) vs No-KL (alpha=0)",
)

SWEEP_ROUND_MAP = {
    "alpha": 4,
    "best_step": 0,
    "kl_p0_tail": 2,
    "kriner_mse": 5,
    "wtp_mae_$": 2,
    "kriner_kl": 3,
    "grummon_end_mse": 5,
    "grummon_end_kl": 3,
}


def collect_alpha_sweep(run_dirs: list[Path]) -> pd.DataFrame:
    if len(run_dirs) == 0:
        raise FileNotFoundError("No alpha-sweep run directories supplied")
    step0 = extract_step0_from_run_dir(run_dirs[0], ("wu2025", "kriner2026", "grummon2025"))
    rows = [
        {
            "run": "Untrained Base (Step 0)",
            "alpha": np.nan,
            "status": "step0",
            "best_step": 0,
            "kl_p0_tail": 0.0,
            "kriner_mse": step0["metrics"]["kriner2026/beta_mse"],
            "wtp_mae_$": wtp_mae_usd(step0["betas_by_validator"]["kriner2026"]),
            "kriner_kl": step0["metrics"]["kriner2026/kl_loss"],
            "grummon_end_mse": step0["metrics"]["grummon2025/beta_mse"],
            "grummon_end_kl": step0["metrics"]["grummon2025/kl_loss"],
        }
    ]
    keyed = []
    for rd in run_dirs:
        info = run_store.read_run_json(rd)
        keyed.append((float(info["identity"]["config"]["kl_anchor_alpha"]), rd, info))
    keyed.sort(key=lambda x: x[0])

    for alpha, rd, info in keyed:
        result = load_paper_run(rd)
        k_metrics, k_betas = result.best_validation["kriner2026"]
        g_pos_mse = result.val_history_df.loc[result.val_history_df.index > 0, "grummon2025/beta_mse"].dropna()
        g_pos_kl = result.val_history_df.loc[result.val_history_df.index > 0, "grummon2025/kl_loss"].dropna()
        rows.append(
            {
                "run": f"alpha = {alpha:g}",
                "alpha": alpha,
                "status": info["status"],
                "best_step": int(result.best["step"]),
                "kl_p0_tail": float(result.step_df["kl_anchor_loss"].dropna().tail(20).mean()),
                "kriner_mse": k_metrics["beta_mse"],
                "wtp_mae_$": wtp_mae_usd(k_betas),
                "kriner_kl": k_metrics["kl_loss"],
                "grummon_end_mse": float(g_pos_mse.iloc[-1]),
                "grummon_end_kl": float(g_pos_kl.iloc[-1]),
            }
        )
    return pd.DataFrame(rows).set_index("run").round(SWEEP_ROUND_MAP)


sweep_800m_df = collect_alpha_sweep(
    [
        *[d for d in (LOCAL_PAPER_RUNS_ROOT / "kl_anchor_800m").iterdir() if (d / "run.json").exists()],
        *[d for d in (PAPER_RUNS_ROOT / "kl_anchor_800m").iterdir() if (d / "run.json").exists()],
    ]
)
sweep_9b_df = collect_alpha_sweep([d for d in (PAPER_RUNS_ROOT / "kl_anchor_9b").iterdir() if (d / "run.json").exists()])

print("=== Pre-Flight KL-Anchor Sweep: kev-0.8b (Wu Table Only, lr=1e-5) ===")
display(sweep_800m_df)
print("=== Pre-Flight KL-Anchor Sweep: kev-9b (Wu Table Only, lr=5e-6) ===")
display(sweep_9b_df)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8))
for ax, df, title in (
    (axes[0], sweep_800m_df.iloc[1:], "kev-0.8b KL Sweep (lr=1e-5)"),
    (axes[1], sweep_9b_df.iloc[1:], "kev-9b KL Sweep (lr=5e-6)"),
):
    x_pos = np.arange(len(df))
    ax.plot(x_pos, df["kriner_mse"], marker="o", linewidth=1.8, label="Kriner best MSE")
    ax.plot(x_pos, df["grummon_end_mse"], marker="^", linewidth=1.6, label="Grummon end MSE")
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{a:g}" for a in df["alpha"]])
    ax.set_xlabel("kl_anchor_alpha")
    ax.set_ylabel("Population Beta MSE")
    ax.set_title(title, fontweight="bold")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5)
fig.tight_layout()
plt.show()

---
## Step 4 of the Story — Hybrid Training: Mixing Accessible Microdata with Published $\beta^*$ Tables (`Row 3` $\to$ `Row 4` vs `Row 5` Full Oracle)

**Claim**: When an analyst holds respondent microdata in one accessible domain (`Berke`, `Row 3`) and supplements it with published regression slopes from a target domain (`Wu` 8 flight slopes, `Row 4`), hybrid training matches—or exceeds—the full `Berke + Wu` microdata oracle (`Row 5`) on the target domain while retaining microdata-grounded out-of-domain generalization:
- **`kev-0.8b` (complete with `test_best`)**: Adding `Wu`'s 8 slopes to `Berke` CE cuts `Kriner` MSE in half (`0.00968 -> 0.00455`, closing **`100.5%`** of the `Row 5` full microdata oracle `0.00462` and matching `$16.45` vs `$16.10` WTP MAE), while preserving held-out `Grummon` (`0.00355`, `60.5%` gap closed) and `COVID` (`0.00442`, **`80.4%`** of the `Row 5` oracle gap closed!). Notice `Row 4 (KL=0.005)` beats `Row 4 (No-KL)` on all three benchmarks (`Kriner`, `Grummon`, `COVID`).
- **`kev-4b` (`Row 4` newly finished; re-run `04-test-best-4b.ipynb` to populate `Row 4` `COVID`)**: Hybrid training cuts `Kriner` MSE `2.2×–3.3×` below `Berke` CE (`0.00100 -> 0.00045` KL / `0.00030` No-KL, WTP MAE down to **`$5.28` / `$4.37`**), surpassing the partial `Row 5` oracle (`0.00073`, `$6.68`) while improving held-out `Grummon` (`0.00704 -> 0.00624` KL / `0.00322` No-KL).

In [ ]:
STEP4_SLUGS = (
    "row1_base_untrained",
    "row3_berke_ce_only",
    "row4_hybrid_berke_ce_wu_table_no_kl",
    "row4_hybrid_berke_ce_wu_table_anchored",
    "row5_full_berke_wu_ce_oracle",
)
print("=== Step 4: Hybrid Microdata (`Berke` CE) + Published Table (`Wu` 8 Slopes) vs Full Oracle (`Row 5` = 100% Gap) ===")
step4_df = build_step_table(("kev-0.8b", "kev-4b"), STEP4_SLUGS, "row5_full_berke_wu_ce_oracle")
display(step4_df)

plot_regime_grid(
    ("kev-0.8b", "kev-4b"),
    (
        "row3_berke_ce_only",
        "row4_hybrid_berke_ce_wu_table_anchored",
        "row4_hybrid_berke_ce_wu_table_no_kl",
        "row5_full_berke_wu_ce_oracle",
    ),
    "Figure 1b — Step 4 Trajectories (Hybrid): Berke CE vs Hybrid (KL / No-KL) vs Full CE Oracle",
)

---
## Step 5a — Substantive Economic & Policy Translation (`Kriner` USD WTP & `COVID` Vaccine Slopes)

1. **`Kriner 2026` Implied Flight Willingness-To-Pay in USD** ($\text{WTP}_k = -\beta_k / \beta_{\text{price per \$}}$): Shows how published-table and hybrid fine-tuning recover dollar valuations across all 9 flight attributes and restore realistic marginal price sensitivity ($d\Pr / \$100$).
2. **`COVID / Kreps et al. 2020` Held-Out Vaccine Preferences (`test_best/`)**: Reveals *why* untrained RLHF models err on policy conjoints (`Row 1` inflates institutional `CDC`/`WHO` endorsement `3×–5×` above human respondents, and unanchored `Row 2 No-KL` inflates it even further on `0.8B`), whereas KL-anchored and microdata/hybrid training dampen endorsement exaggeration toward human trade-offs.

In [ ]:
STORY_ORDER_SLUGS = (
    "row1_base_untrained",
    "row2_ablation_wu_table_no_kl",
    "row2_wu_table_anchored",
    "row2b_wu_ce_oracle",
    "row3_berke_ce_only",
    "row4_hybrid_berke_ce_wu_table_no_kl",
    "row4_hybrid_berke_ce_wu_table_anchored",
    "row5_full_berke_wu_ce_oracle",
)

for model_slug in MODEL_SLUGS:
    betas_map = kriner_betas_by_model[model_slug]
    regimes_df = regimes_by_model[model_slug]
    first_df = next(iter(betas_map.values()))
    human_wtp = kriner_wtp_from_series(first_df["human_pop_beta"])
    cols = {"Human (OLS)": human_wtp}

    for slug in STORY_ORDER_SLUGS:
        if slug not in betas_map:
            continue
        col_name = SHORT_COL_LABELS[slug]
        st = str(regimes_df.loc[slug, "status"])
        if st not in ("finished", "step0"):
            col_name = f"{col_name}*"
        cols[col_name] = kriner_wtp_from_series(betas_map[slug]["model_pop_beta"])

    wtp_df = pd.DataFrame(cols)
    wtp_df.loc["MAE vs Human WTP ($)"] = wtp_df.iloc[:-1].sub(wtp_df["Human (OLS)"].iloc[:-1], axis=0).abs().mean()
    print(f"=== {model_slug} — Kriner 2026 Implied Flight WTP in USD (* = incomplete run best.pt) ===")
    display(wtp_df.round(2))

for model_slug in ("kev-0.8b", "kev-4b"):
    c_map = covid_betas_by_model[model_slug]
    if len(c_map) == 0:
        continue
    first_c = next(iter(c_map.values()))
    c_cols = {"Human (OLS)": first_c["human_pop_beta"]}
    for slug in STORY_ORDER_SLUGS:
        if slug not in c_map:
            continue
        c_cols[SHORT_COL_LABELS[slug]] = c_map[slug]["model_pop_beta"]
    covid_term_df = pd.DataFrame(c_cols)
    covid_term_df.loc["Slope MSE vs Human"] = (
        covid_term_df.sub(covid_term_df["Human (OLS)"], axis=0).pow(2).mean()
    )
    print(f"=== {model_slug} — Held-Out COVID Vaccine Conjoint (`Kreps 2020`) Population Betas (`test_best/`) ===")
    display(covid_term_df.round(4))

---
## Step 5b — Unified End-to-End Story Progression (`Row 1` $\to$ `Row 5` vs Global `Row 5` Oracle)

All regimes ordered sequentially along the paper's narrative (`Row 1` Base $\to$ `Row 2` Table No-KL / KL $\to$ `Row 2b` Study CE $\to$ `Row 3` `Berke` CE $\to$ `Row 4` Hybrid No-KL / KL $\to$ `Row 5` Full CE Oracle), accompanied by a visual cross-benchmark summary across `Kriner` (Validation), `Grummon` (Held-Out Test 1), and `COVID` (Held-Out Test 2).

In [ ]:
print("=== Master Story Progression (`kev-0.8b` & `kev-4b`, Gap % Scaled to Global `Row 5` Full CE Oracle) ===")
master_df = build_step_table(("kev-0.8b", "kev-4b"), STORY_ORDER_SLUGS, "row5_full_berke_wu_ce_oracle")
display(master_df)

PLOT_BENCHMARKS = [
    ("kriner_mse", "Kriner 2026 MSE (Flights — Validation)"),
    ("grummon_mse", "Grummon 2025 MSE (Food Labels — Test)"),
    ("covid_mse", "COVID / Kreps 2020 MSE (Vaccines — Test)"),
]
BAR_COLORS = {
    "row1_base_untrained": "#8c8c8c",
    "row2_ablation_wu_table_no_kl": "#d95f02",
    "row2_wu_table_anchored": "#1b9e77",
    "row2b_wu_ce_oracle": "#66a61e",
    "row3_berke_ce_only": "#7570b3",
    "row4_hybrid_berke_ce_wu_table_no_kl": "#e7298a",
    "row4_hybrid_berke_ce_wu_table_anchored": "#2b8cbe",
    "row5_full_berke_wu_ce_oracle": "#252525",
}

fig, axes = plt.subplots(2, 3, figsize=(15.5, 7.2), sharey=False)
for r_idx, model_slug in enumerate(("kev-0.8b", "kev-4b")):
    sub = regimes_by_model[model_slug].loc[list(STORY_ORDER_SLUGS)]
    x = np.arange(len(STORY_ORDER_SLUGS))
    short_labels = [SHORT_COL_LABELS[s].replace(": ", ":\n") for s in STORY_ORDER_SLUGS]
    for c_idx, (col, title) in enumerate(PLOT_BENCHMARKS):
        ax = axes[r_idx, c_idx]
        vals = sub[col].to_numpy(dtype=float)
        colors = [BAR_COLORS[s] for s in STORY_ORDER_SLUGS]
        bars = ax.bar(x, np.nan_to_num(vals, nan=0.0), color=colors, width=0.68, edgecolor="black", linewidth=0.5)
        for idx_b, v in enumerate(vals):
            if np.isnan(v):
                ax.text(x[idx_b], 0.0, "pending", rotation=90, ha="center", va="bottom", fontsize=7.5, color="#555555")
        ax.axhline(float(sub.loc["row1_base_untrained", col]), color="#8c8c8c", linestyle=":", linewidth=1.1)
        if r_idx == 0:
            ax.set_title(title, fontsize=10.5, fontweight="bold")
        if c_idx == 0:
            ax.set_ylabel(f"{model_slug}\nPopulation Beta MSE", fontsize=10, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(short_labels if r_idx == 1 else [""] * len(x), rotation=35, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.25)

fig.suptitle("End-to-End Story Progression Across Validation & Held-Out Test Domains (Lower MSE = Better)", fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

---
## Appendix — Pre-Paper Reference Runs (`v8`, `v12`, `v13`, `v14`)

Cross-check against earlier exploratory runs (`v8`/`v12`/`v14` used `lr = 4e-5`, `1` epoch; `v13` used `alpha = 0`, `5` epochs).

In [ ]:
LEGACY_SPECS = [
    ("kev-0.8b", "v13 Wu Table-Only (No-KL, ep=5, lr=1e-5)", LEGACY_RUNS_ROOT / "train-beta-v13-wu-beta-only-800m" / "beta_only_split_lr=1e-05_S=40_ep=5", load_population_run),
    ("kev-0.8b", "v8 Full CE (Wu+Berke, ep=1, lr=4e-5)", LEGACY_RUNS_ROOT / "train-beta-v8-wu-berke-sample-lrscale-800m" / "lambda=0_lr=4e-05_S=40_ep=1", load_run),
    ("kev-4b", "v12 Full CE (Wu+Berke, ep=1, lr=4e-5)", LEGACY_RUNS_ROOT / "train-beta-v12-wu-berke-scale-4b" / "lambda=0_lr=4e-05_S=40_ep=1", load_run),
    ("kev-4b", "v14 Full CE + Pop-Beta (lambda=0.25, lr=4e-5)", LEGACY_RUNS_ROOT / "train-beta-v14-wu-berke-ce-population-4b" / "lambda=0.25_lr=4e-05_S=40_ep=1", load_ce_population_run),
    ("kev-4b", "v14 Full CE + Pop-Beta (lambda=1.0, lr=4e-5)", LEGACY_RUNS_ROOT / "train-beta-v14-wu-berke-ce-population-4b" / "lambda=1_lr=4e-05_S=40_ep=1", load_ce_population_run),
]

legacy_rows = []
seen_models: set[str] = set()
for model_slug, label, run_dir, loader in LEGACY_SPECS:
    if not (run_dir / "run.json").exists():
        continue
    run_info = run_store.read_run_json(run_dir)
    if run_info["status"] != "finished":
        continue
    if model_slug not in seen_models:
        seen_models.add(model_slug)
        base = extract_step0_from_run_dir(run_dir, VALIDATORS)
        legacy_rows.append(
            {
                "model": model_slug,
                "reference_run": "Untrained Base (Step 0)",
                "best_step": 0,
                "kriner_mse": base["metrics"]["kriner2026/beta_mse"],
                "wtp_mae_$": wtp_mae_usd(base["betas_by_validator"]["kriner2026"]),
                "kriner_kl": base["metrics"]["kriner2026/kl_loss"],
                "grummon_mse": base["metrics"]["grummon2025/beta_mse"],
                "grummon_kl": base["metrics"]["grummon2025/kl_loss"],
            }
        )
    result = loader(run_dir)
    legacy_rows.append(
        {
            "model": model_slug,
            "reference_run": label,
            "best_step": int(result.best["step"]),
            "kriner_mse": float(result.best_row["kriner2026/beta_mse"]),
            "wtp_mae_$": wtp_mae_usd(result.best_validation["kriner2026"][1]),
            "kriner_kl": float(result.best_row["kriner2026/kl_loss"]),
            "grummon_mse": float(result.best_row["grummon2025/beta_mse"]),
            "grummon_kl": float(result.best_row["grummon2025/kl_loss"]),
        }
    )

legacy_df = pd.DataFrame(legacy_rows).set_index(["model", "reference_run"]).round(ROUND_MAP)
display(legacy_df)